## Application of Machine Learning To Epileptic Seizure Detection
Ali Shoeb and John Guttag

In [1]:
!pip install pyedflib

In [2]:
# Libraries
import numpy as np
import random
from sklearn import svm
from sklearn.model_selection import StratifiedKFold
import pickle
from sklearn.metrics import confusion_matrix
from Splits import Splits

## Data preparation

In [3]:
# Getting EEG data: alternative 1
splits = Splits.patient_splits(patient_id = 10, patient_path = './chb-mit-scalp-eeg-database-1.0.0/chb10/')

Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_01.edf
Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_02.edf
Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_03.edf
Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_04.edf
Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_05.edf
Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_06.edf
Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_07.edf
Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_08.edf
Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_12.edf
Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_13.edf
Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_14.edf
Reading EEG data from file ./chb-mit-scalp-eeg-database-1.0.0/chb10\chb10_15.edf
Reading EEG data from file .

In [4]:
# Taking random sample for training: alternative 2
idx_train_sample = random.sample(range(0, len(splits[0][0])), k = 50000)
X_train_sample = (splits[0][0][idx_train_sample])
y_train_sample = (splits[0][1][idx_train_sample])

# Checking the prior is stable
sum(splits[0][1])/len(splits[0][1]), sum(y_train_sample)/len(y_train_sample)

(0.15966386554621848, 0.15778)

In [5]:
# Taking random sample for testing: alternative 2
idx_test_sample = random.sample(range(0, len(splits[1][0])), k = 20000)
X_test_sample = splits[1][0][idx_test_sample]
y_test_sample = splits[1][1][idx_test_sample]

## Training

In [6]:
# Defining SVM
rbf_svc = svm.SVC(kernel='rbf', C=1.0, gamma=0.1)

In [8]:
# Cross-validation
skf = StratifiedKFold(n_splits=3)

for i, (train_index, test_index) in enumerate(skf.split(X_train_sample, y_train_sample)):
    print('Split nº: ', str(i+1))
    # Fit model
    rbf_svc.fit(X_train_sample[train_index], y_train_sample[train_index])
    
    # Predict
    predict_sample = rbf_svc.predict(X_train_sample[test_index])
    tn, fp, fn, tp = confusion_matrix(y_train_sample[test_index], predict_sample).ravel()
    print('\tSensitivity: ', str(tp/(tp + fn)))
    print('\tSpecificity: ', str(tn/(tn + fp)))

Split nº:  1
	Sensitivity:  0.00494296577946768
	Specificity:  1.0
Split nº:  2
	Sensitivity:  0.006844106463878327
	Specificity:  1.0
Split nº:  3
	Sensitivity:  0.00798782807151008
	Specificity:  1.0


In [ ]:
with open('./rbf_svc.pkl', 'wb') as f:
    pickle.dump(rbf_svc, f)

## Testing

In [9]:
predict_sample = rbf_svc.predict(X_test_sample)

tn, fp, fn, tp = confusion_matrix(y_test_sample, predict_sample).ravel()
print('Results in test:')
print('\tSensitivity: ', str(tp/(tp + fn)))
print('\tSpecificity: ', str(tn/(tn + fp)))

Results in test:
	Sensitivity:  0.0
	Specificity:  1.0
